In [ ]:
import polars as pl

In [ ]:
# Lazy load all tables
aisles = scan_delta("aisles")
departments = scan_delta("departments")
prior = scan_delta("order_products__prior")
train = scan_delta("order_products__train")
orders = scan_delta("orders")
products = scan_delta("products")

In [ ]:
# Join prior orders with order metadata
prior_full = (
    prior
    .join(orders, on="order_id", how="inner")
)

In [ ]:
# Aggregate user-product interaction history
user_product_features = (
    prior_full
    .group_by(["user_id", "product_id"])
    .agg([
        pl.len().alias("times_ordered"),
        pl.mean("add_to_cart_order").alias("avg_cart_position"),
        pl.max("order_number").alias("last_order_number"),
        pl.mean("reordered").alias("reorder_rate"),
    ])
)

In [ ]:
# Add product-level info
product_features = (
    products
    .join(departments.lazy(), on="department_id", how="left")
    .join(aisles.lazy(), on="aisle_id", how="left")
)

user_product_with_meta = (
    user_product_features
    .join(product_features.select(["product_id", "product_name", "department", "aisle"]), on="product_id", how="left")
)

In [ ]:
# Join with training set (labels)

train_orders = (
    orders
    .filter(pl.col("eval_set") == "train")
    .select(["order_id", "user_id"])
)

train_labels = (
    train_orders
    .join(train, on="order_id", how="inner")
    .select(["user_id", "product_id", "reordered"])
)

In [ ]:
# Compute final dataset

final_features_df = (
    user_product_with_meta
    .join(train_labels, on=["user_id", "product_id"], how="left")
    .with_columns(pl.col("reordered").fill_null(0))
    .collect()
)

In [ ]:
# Write final dataset to delta table

write_delta(final_features_df, "final_features", mode="overwrite")